# GeoProspectNet — Kaggle full pipeline

**Runs every experiment from raw data → trained models → 13 evaluation tracks → 20 figures → reviewer-response card, in a single 9-hour Kaggle GPU session.**

**Robustness contract:**
- Every cell has a try/except boundary; failure in one section never aborts the rest of the pipeline.
- Each step caches outputs to `outputs/`; reruns short-circuit completed work.
- Data pulls fall back to whichever public source is reachable.
- Missing optional dependencies (`cartopy`) skip the corresponding cell rather than crashing.
- A preflight cell (§0) tells you in 30 seconds whether the environment is ready.

**To run on Kaggle:**
1. New Notebook → Settings: Accelerator = GPU P100 (or T4×2), Persistence = Variables + Files, Internet = On
2. Upload the GitHub repo as a dataset, *or* let the §1 clone cell pull it from `keshavkrishnan08/Geothermal`
3. Run-All

Cold rerun (no caches): ~6-8 h on GPU. Warm rerun (caches present): ~30 min.

## §0  Preflight — 30-second health check

In [ ]:
import os, sys, json, shutil, subprocess, time, traceback
from pathlib import Path

# Where to operate
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'Geothermal2'
OUT  = REPO / 'outputs'
DATA = REPO / 'data'

# A simple step runner that never aborts the whole notebook
def run(name, cmd, timeout=3600, cwd=None):
    """Run a subprocess; print PASS / FAIL with elapsed; never raise."""
    t0 = time.time()
    print(f'>> {name}', flush=True)
    try:
        r = subprocess.run(cmd, cwd=str(cwd or REPO), capture_output=True,
                           text=True, timeout=timeout)
        ms = (time.time() - t0) * 1000
        if r.returncode == 0:
            print(f'   ✓  ({ms:6.0f} ms)')
            return True
        print(f'   ✗  exit {r.returncode}  ({ms:6.0f} ms)')
        print(f'   stderr tail:\n{r.stderr[-500:]}')
        return False
    except subprocess.TimeoutExpired:
        print(f'   ✗  TIMEOUT after {timeout}s')
        return False
    except Exception as e:
        print(f'   ✗  exception: {e}')
        return False

def safe(name, fn):
    """Wrap a Python callable in try/except — never abort the notebook."""
    t0 = time.time()
    print(f'>> {name}', flush=True)
    try:
        fn()
        print(f'   ✓  ({(time.time() - t0)*1000:6.0f} ms)')
        return True
    except Exception as e:
        print(f'   ✗  {type(e).__name__}: {e}')
        traceback.print_exc(limit=2)
        return False

# Python + torch + cuda
import importlib
print(f'python: {sys.version.split()[0]}')
for mod in ['numpy', 'pandas', 'torch', 'sklearn', 'scipy', 'matplotlib',
            'rasterio', 'xgboost']:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '?')
        print(f'  {mod:12s} {v}')
    except ImportError:
        print(f'  {mod:12s} MISSING (will install in §1)')

try:
    import torch
    print(f'  cuda available: {torch.cuda.is_available()}, '
          f'device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')
except ImportError:
    pass
print(f'  workspace: {WORK}')
print(f'  free disk MB: ', end='')
try:
    import shutil as _sh
    print(f'{_sh.disk_usage(WORK).free // (1024**2):,}')
except Exception:
    print('unknown')

## §1  Install missing deps + clone repo

If the Kaggle environment is missing anything, install it. If `Geothermal2/src/` isn't already on disk (e.g. uploaded as a Kaggle dataset), clone from GitHub.

In [ ]:
DEPS = ['xgboost', 'rasterio', 'scipy', 'scikit-learn', 'pyyaml']
for pkg in DEPS:
    try:
        importlib.import_module(pkg.replace('-', '_').replace('scikit_learn', 'sklearn'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', pkg])
        print(f'  installed {pkg}')

# Optional: cartopy for state-outline maps (failure is non-fatal — fig1 falls back)
try:
    import cartopy
    print(f'  cartopy {cartopy.__version__}')
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'cartopy'])
    except subprocess.CalledProcessError:
        print('  cartopy install failed — fig1 will use plain matplotlib (non-fatal)')

REPO_URL = 'https://github.com/keshavkrishnan08/Geothermal.git'
if not (REPO / 'src').exists():
    # Try git clone; if that fails, look for a Kaggle-dataset copy
    REPO.mkdir(parents=True, exist_ok=True)
    rc = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)],
                        capture_output=True, text=True).returncode
    if rc != 0:
        print('  git clone failed — looking for a Kaggle-dataset copy')
        for cand in [Path('/kaggle/input/geothermal'), Path('/kaggle/input/geothermal2'),
                      Path('/kaggle/input/geoprospectnet')]:
            if (cand / 'src').exists():
                print(f'  using {cand}')
                # Copy code only — large data files are .gitignored anyway
                for sub in ['src', 'configs', 'notebooks', 'manuscript']:
                    s = cand / sub
                    if s.exists():
                        shutil.copytree(s, REPO / sub, dirs_exist_ok=True)
                break
        else:
            raise RuntimeError('No src/ found. Upload the repo as a Kaggle dataset or fix REPO_URL.')

os.chdir(REPO)
sys.path.insert(0, str(REPO))
for p in [OUT/'results', OUT/'checkpoints', OUT/'figures', OUT/'maps',
          DATA/'raw', DATA/'processed', DATA/'metadata',
          DATA/'processed/splits']:
    p.mkdir(parents=True, exist_ok=True)
print(f'  repo ready at {REPO}')
print(f'  src present: {(REPO / "src").exists()},  configs present: {(REPO / "configs").exists()}')

## §2  Raw data ingest

Pulls every public source (gravity, magnetic, SMU heat-flow, SRTM, SGMC lithology, QFaults, GEOTHERM, Williams 2008 inventory, NV permits, Mordensky 2023). The `src.data.download` script is idempotent — if a file exists it's not re-fetched.

If you uploaded the data as a Kaggle dataset under `/kaggle/input/geothermal-data`, set `DATA_INPUT` and we'll symlink to it instead.

In [ ]:
DATA_INPUT = None  # set to a /kaggle/input/<dataset> path to skip the download
if DATA_INPUT and Path(DATA_INPUT).exists():
    for sub in ['raw', 'processed', 'metadata']:
        s = Path(DATA_INPUT) / sub
        if s.exists():
            for f in s.rglob('*'):
                if f.is_file():
                    dst = DATA / sub / f.relative_to(s)
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    if not dst.exists():
                        try: dst.symlink_to(f)
                        except Exception: shutil.copy(f, dst)
    print(f'  symlinked from {DATA_INPUT}')

if not (DATA / 'raw/geophysics/bouguer_gravity.tif').exists():
    run('Pull raw public data', [sys.executable, '-m', 'src.data.download'],
        timeout=3600)
else:
    print('  raw data already present — skipping download')

## §3  Process modalities → 235 K cell × 6-channel arrays

Runs in this order: geophysics (gravity / mag / heat flow / SRTM) → geochemistry (GEOTHERM springs) → geology (SGMC lithology + Mordensky faults). The output is `data/processed/*.npy`.

**Bottleneck:** `geophysics_patches.npy` is 5.4 GB. Skip the rebuild if the file already exists.

In [ ]:
PROCESSED = DATA / 'processed'
if not (PROCESSED / 'geophysics_patches.npy').exists():
    run('process_geophysics', [sys.executable, '-m', 'src.data.process_geophysics',
                                '--config', 'configs/cpu_max.yaml'], timeout=3600)
else:
    print('  geophysics_patches.npy cached')
if not (PROCESSED / 'geochemistry_features.npy').exists():
    run('process_geochemistry', [sys.executable, '-m', 'src.data.process_geochemistry',
                                  '--config', 'configs/cpu_max.yaml'], timeout=1800)
else:
    print('  geochemistry_features.npy cached')
if not (PROCESSED / 'geology_features.npy').exists():
    run('process_geology', [sys.executable, '-m', 'src.data.process_geology',
                             '--config', 'configs/cpu_max.yaml'], timeout=3600)
else:
    print('  geology_features.npy cached')
if not (PROCESSED / 'labels.npy').exists():
    run('build_labels', [sys.executable, '-m', 'src.data.build_labels',
                          '--config', 'configs/cpu_max.yaml'], timeout=900)
if not (PROCESSED / 'splits/train_val_test.json').exists():
    run('build_splits', [sys.executable, '-m', 'src.data.build_splits',
                          '--config', 'configs/cpu_max.yaml'], timeout=900)
print('  data ready')

## §4  Train 4 model variants  (5–60 min each on GPU)

Train cpu_calibrated, cpu_tuned, cpu_margin, cpu_max in sequence. Each one is skipped if its named checkpoint already exists.

In [ ]:
CONFIGS = ['cpu_calibrated', 'cpu_tuned', 'cpu_margin', 'cpu_max']
for name in CONFIGS:
    named_ckpt = OUT / f'checkpoints/random_{name}_seed42.pt'
    if named_ckpt.exists():
        print(f'  ✓ {name} cached  ({named_ckpt.stat().st_size // 1024} KB)')
        continue
    # Rebuild labels + splits for THIS config's negative ratio (avoids cached-splits bug)
    if name != 'cpu_max':
        run(f'build_labels[{name}]', [sys.executable, '-m', 'src.data.build_labels',
                                       '--config', f'configs/{name}.yaml'], timeout=900)
        run(f'build_splits[{name}]', [sys.executable, '-m', 'src.data.build_splits',
                                       '--config', f'configs/{name}.yaml'], timeout=900)
    ok = run(f'train {name}',
             [sys.executable, '-m', 'src.training.train',
              '--config', f'configs/{name}.yaml', '--random', '--seed', '42'],
             timeout=7200)
    if ok:
        src_ck = OUT / 'checkpoints/random_seed42_best.pt'
        if src_ck.exists():
            shutil.copy(src_ck, named_ckpt)
            print(f'  saved {named_ckpt.name}')

## §5  Multi-seed sensitivity  (cpu_margin × {42, 7, 13, 21})

In [ ]:
run('multi_seed sweep',
    [sys.executable, '-m', 'src.training.multi_seed',
     '--config', 'configs/cpu_margin.yaml',
     '--seeds', '42', '7', '13', '21'], timeout=7200)
p = OUT / 'results/multi_seed_sensitivity.csv'
if p.exists():
    import pandas as pd
    print(pd.read_csv(p).to_string(index=False, float_format='%.2f'))

## §6  Architecture ablation  (no-contrastive / no-spatial / no-attention)

In [ ]:
run('arch ablation',
    [sys.executable, '-m', 'src.training.tune_separation',
     '--configs',
     'configs/cpu_margin_no_contrastive.yaml',
     'configs/cpu_margin_no_spatial.yaml',
     'configs/cpu_margin_no_attention.yaml'], timeout=7200)
p = OUT / 'results/separation_sweep.csv'
if p.exists():
    import pandas as pd
    df = pd.read_csv(p)
    print(df[['config', 'holdout_mean_pct', 'separation_gap']]
          .to_string(index=False, float_format='%.2f'))
    shutil.copy(p, OUT / 'results/separation_sweep_arch_ablation.csv')

## §7  Negative-pool size sweep  (with rebuilt splits per ratio — the bug-fixed version)

In [ ]:
import yaml, tempfile
for ratio in [3, 5, 10, 20]:
    named = OUT / f'checkpoints/random_cpu_margin_neg{ratio}_seed42.pt'
    if named.exists():
        print(f'  ratio {ratio}: cached')
        continue
    base = yaml.safe_load(open(REPO / 'configs/cpu_margin.yaml'))
    base['labels']['negative_pos_ratio'] = ratio
    tmp = Path(tempfile.mkdtemp()) / f'cpu_margin_neg{ratio}.yaml'
    tmp.write_text(yaml.safe_dump(base))
    run(f'rebuild labels @ neg{ratio}', [sys.executable, '-m', 'src.data.build_labels',
                                          '--config', str(tmp)], timeout=900)
    run(f'rebuild splits @ neg{ratio}', [sys.executable, '-m', 'src.data.build_splits',
                                          '--config', str(tmp)], timeout=900)
    if run(f'train cpu_margin neg{ratio}', [sys.executable, '-m', 'src.training.train',
                                              '--config', str(tmp), '--random', '--seed', '42'],
            timeout=7200):
        src_ck = OUT / 'checkpoints/random_seed42_best.pt'
        if src_ck.exists():
            shutil.copy(src_ck, named)
# Restore canonical labels for downstream cells
run('restore canonical labels (cpu_max)', [sys.executable, '-m', 'src.data.build_labels',
                                            '--config', 'configs/cpu_max.yaml'], timeout=900)
run('restore canonical splits (cpu_max)', [sys.executable, '-m', 'src.data.build_splits',
                                            '--config', 'configs/cpu_max.yaml'], timeout=900)

## §8  LOFCV  (5 quick folds, cpu_max)

In [ ]:
run('LOFCV quick', [sys.executable, '-m', 'src.training.train',
                      '--config', 'configs/cpu_max.yaml', '--quick', '--seed', '42'],
    timeout=7200)

## §9  Continental inference  (cpu_max) → score every cell

In [ ]:
def cache_scores():
    import numpy as np, torch, yaml
    from src.data.dataset import GeoProspectDataset, make_loader
    from src.models.geoprospectnet import GeoProspectNet
    scores_path = OUT / 'results/scores_cpu_max.npy'
    if scores_path.exists():
        print(f'  scores cached  ({scores_path.stat().st_size // 1024} KB)')
        return
    cfg = yaml.safe_load(open(REPO / 'configs/cpu_max.yaml'))
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ck = torch.load(OUT / 'checkpoints/random_cpu_max_seed42.pt', map_location=device, weights_only=False)
    model = GeoProspectNet(cfg).to(device); model.load_state_dict(ck['state_dict']); model.eval()
    n = int(np.load(DATA / 'processed/labels.npy', mmap_mode='r').shape[0])
    ds = GeoProspectDataset(DATA / 'processed', indices=np.arange(n), use_thermal=False)
    loader = make_loader(ds, batch_size=512, shuffle=False, num_workers=0)
    out=[]
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) for k, v in b.items()}
            out.append(torch.sigmoid(model(b)['logits']).cpu().numpy())
    import numpy as _np
    _np.save(scores_path, _np.concatenate(out).astype('float32'))
    print(f'  scored {n:,} cells')
safe('continental inference', cache_scores)

## §10  Validation cohorts  (NC1 + NC3 + temporal hold-out)

In [ ]:
ok = run('negative_controls', [sys.executable, '-m', 'src.evaluation.negative_controls',
                                 '--config', 'configs/cpu_max.yaml',
                                 '--ckpt', 'outputs/checkpoints/random_cpu_max_seed42.pt'],
          timeout=900)
if ok and (OUT / 'results/negative_controls.csv').exists():
    shutil.copy(OUT / 'results/negative_controls.csv',
                OUT / 'results/negative_controls_cpu_max.csv')

## §11  Hard-negative cohorts  NC4 / NC5 / NC6

In [ ]:
run('hard_negatives', [sys.executable, '-m', 'src.evaluation.hard_negatives'], timeout=1800)

## §12  Classical baselines + Mordensky 2023 head-to-head

In [ ]:
run('baselines', [sys.executable, '-m', 'src.evaluation.baselines',
                   '--config', 'configs/cpu_max.yaml'], timeout=1800)
if (DATA / 'raw/labels/mordensky2023_full.csv').exists():
    run('mordensky comparison', [sys.executable, '-m', 'src.evaluation.mordensky_comparison'],
        timeout=600)
else:
    print('  mordensky2023_full.csv missing — skipping head-to-head')

## §13  Modality permutation importance + attention-by-province

In [ ]:
run('modality_analysis', [sys.executable, '-m', 'src.evaluation.modality_analysis',
                            '--config', 'configs/cpu_max.yaml',
                            '--ckpt', 'outputs/checkpoints/random_cpu_max_seed42.pt',
                            '--n_repeats', '5'], timeout=3600)

## §14  Calibration — Brier, ECE, reliability diagram

In [ ]:
run('calibration', [sys.executable, '-m', 'src.evaluation.calibration',
                     '--config', 'configs/cpu_max.yaml',
                     '--scores', 'outputs/results/scores_cpu_max.npy'], timeout=300)

## §15  Discovery pipeline → 33 consensus sites

In [ ]:
for cfg_name, thr in [('cpu_margin', 0.9994), ('cpu_max', 0.9962)]:
    ckpt = OUT / f'checkpoints/random_{cfg_name}_seed42.pt'
    if not ckpt.exists():
        print(f'  skipping {cfg_name} discovery — checkpoint missing')
        continue
    shutil.copy(ckpt, OUT / 'checkpoints/discovery_full.pt')
    if run(f'discovery {cfg_name}', [sys.executable, '-m', 'src.evaluation.discovery',
                                       '--config', f'configs/{cfg_name}.yaml',
                                       '--threshold', str(thr),
                                       '--uncertainty', '0.50',
                                       '--min_samples', '3'], timeout=3600):
        if (OUT / 'results/table3_discoveries.csv').exists():
            shutil.copy(OUT / 'results/table3_discoveries.csv',
                        OUT / f'results/table3_discoveries_{cfg_name}.csv')

def consensus():
    import numpy as np, pandas as pd
    from scipy.spatial import cKDTree
    cm_path = OUT / 'results/table3_discoveries_cpu_margin.csv'
    cx_path = OUT / 'results/table3_discoveries_cpu_max.csv'
    if not (cm_path.exists() and cx_path.exists()):
        print('  missing one or both discovery CSVs')
        return
    cm = pd.read_csv(cm_path); cx = pd.read_csv(cx_path)
    cm = cm[cm.plausibility == 'plausible'].reset_index(drop=True)
    cx = cx[cx.plausibility == 'plausible'].reset_index(drop=True)
    if len(cm) == 0 or len(cx) == 0:
        print('  no plausible discoveries to merge')
        return
    lat0 = float(cm.lat.mean()); cos_lat0 = float(np.cos(np.radians(lat0)))
    xy_m = np.column_stack([cm.lon * 111.32 * cos_lat0, cm.lat * 111.32])
    xy_x = np.column_stack([cx.lon * 111.32 * cos_lat0, cx.lat * 111.32])
    d, _ = cKDTree(xy_x).query(xy_m, k=1)
    consensus = cm[d < 25].reset_index(drop=True)
    consensus.to_csv(OUT / 'results/consensus_discoveries.csv', index=False)
    print(f'  consensus: {len(consensus)} of {len(cm)} (cpu_margin) ∩ {len(cx)} (cpu_max)')
safe('build consensus', consensus)

## §16  MWe v2 — temperature-dependent η + Monte Carlo (2 000 draws/site)

In [ ]:
run('mwe v2', [sys.executable, '-m', 'src.evaluation.mwe_estimation_v2',
                '--in_csv', 'outputs/results/consensus_discoveries.csv',
                '--out_csv', 'outputs/results/consensus_with_mwe_v2.csv',
                '--n_samples', '2000'], timeout=600)

## §17  Per-discovery audits — NV permits + reservoir thickness + LOFCV buffer

In [ ]:
run('field validation',     [sys.executable, '-m', 'src.evaluation.field_validation'],    timeout=300)
run('reservoir thickness',  [sys.executable, '-m', 'src.evaluation.reservoir_thickness'], timeout=600)
run('LOFCV-buffer audit',   [sys.executable, '-m', 'src.evaluation.lofcv_buffer_audit'],  timeout=600)

## §18  Economic + policy framing — LCOE / GeoVision / Earthshot / CO₂

In [ ]:
run('economic_analysis', [sys.executable, '-m', 'src.evaluation.economic_analysis'], timeout=120)

## §19  Pre-registration manifest — SHA-256, frozen timestamp

In [ ]:
run('preregister', [sys.executable, '-m', 'src.evaluation.preregister'], timeout=120)

## §20  OOD eastern-US test — 535 K cells, streaming inference

In [ ]:
run('ood eastern', [sys.executable, '-m', 'src.evaluation.ood_eastern_us'], timeout=3600)

## §21  Generate all 20 publication figures

In [ ]:
run('figures', [sys.executable, '-m', 'src.visualization.make_figures'], timeout=600)
figs = sorted((OUT / 'figures').glob('fig*.png'))
print(f'  {len(figs)} PNG figures present')

## §22  Final reviewer-response summary card

In [ ]:
def summary_card():
    import json, pandas as pd
    def _read(path, comment=None):
        return pd.read_csv(path, comment=comment) if comment else pd.read_csv(path)
    nc       = _read(OUT / 'results/negative_controls_cpu_max.csv')
    mwe      = _read(OUT / 'results/consensus_with_mwe_v2.csv')
    bl       = _read(OUT / 'results/baselines.csv')
    ms       = _read(OUT / 'results/multi_seed_sensitivity.csv')
    econ     = json.load(open(OUT / 'results/economic_analysis.json'))
    manifest = json.load(open(OUT / 'results/preregistration_manifest.json'))
    east     = _read(OUT / 'results/ood_eastern_us_validation.csv')
    print('=' * 66)
    print('              GeoProspectNet — final summary card')
    print('=' * 66)
    print(f'Cells scored                       : 235,470')
    print(f'Hold-out top-10 capture            : {nc.loc[nc.cohort=="temporal_holdout_post2008","frac_above_90pct"].iat[0]:.0%}')
    print(f'Separation gap (multi-seed)        : {ms.gap.mean():.1f} ± {ms.gap.std():.1f} pp')
    print(f'Consensus discoveries              : {len(mwe)}')
    print(f'Cumulative MWe (P50)               : {mwe.mwe_p50.sum():,.0f}')
    print(f'MWe P10 / P90                      : {mwe.mwe_p10.sum():,.0f} / {mwe.mwe_p90.sum():,.0f}')
    print(f'Capacity multiplier vs US 2023     : {mwe.mwe_p50.sum()/3800:.2f}×')
    print(f'LCOE baseline                      : ${econ["lcoe_baseline_per_mwh"]:.0f}/MWh')
    print(f'GeoVision 2050 fraction (P50)      : {100*econ["geovision_fraction_p50"]:.1f}%')
    print(f'30-yr CO₂ displaced (P50)          : {econ["co2_displaced_Mt_30yr_p50"]:.0f} Mt')
    print(f'Pre-registration SHA-256           : {manifest["sha256"][:24]}...')
    print(f'OOD eastern mean percentile        : {east[east.lon > -100].percentile.mean():.1f}  (honest disclosure)')
    print('Baselines on hold-out (overfitting test):')
    print(bl[['method','holdout_mean_pct','holdout_top10_capture']]
          .to_string(index=False, float_format='%.2f'))
safe('summary card', summary_card)

---
**Done.** All artifacts are under `outputs/`:
- `outputs/results/*.csv` — 27 result tables
- `outputs/checkpoints/*.pt` — trained models (multi-seed + ablation variants)
- `outputs/figures/*.{png,pdf}` — 20 publication-grade figures with unified palette
- `outputs/maps/prospectivity.tif` — continental GeoTIFF (open in QGIS)

Download a zip of `outputs/` from the Kaggle file panel when the kernel finishes.